# Oil Palm Disease Classification using ALOS-2 SAR

**Postgraduate Research** — Prerequisites & Environment Setup

This notebook sets up the full environment for processing ALOS-2 PALSAR-2 data:
- **ESA SNAP 13.0.0** (Sentinel Toolboxes) — SAR processing engine
- **esa_snappy** — Python bridge to SNAP's Java API
- **Scientific Python stack** — numpy, rasterio, scikit-learn, etc.

---

## Prerequisites

### 1. System Setup — Python Environment + Dependencies

Installs `uv` (fast package manager), creates a virtual environment, and installs all Python packages.

In [ ]:
# System checks
import platform, sys, subprocess, shutil, os
print(f"Platform: {platform.system()} {platform.release()}")
print(f"Architecture: {platform.machine()}")
print(f"Python: {sys.version}")
print(f"JAVA_HOME: {os.environ.get('JAVA_HOME', 'not set')}")
print(f"wget: {shutil.which('wget') or 'not found'}")
print(f"curl: {shutil.which('curl') or 'not found'}")
print(f"java: {shutil.which('java') or 'not found (SNAP bundles its own JRE)'}")

In [ ]:
%%bash
set -e

# Install uv if missing
if ! command -v uv &>/dev/null; then
    echo "Installing uv..."
    curl -LsSf https://astral.sh/uv/install.sh | sh
    source "$HOME/.cargo/env"
fi
echo "uv: $(uv --version)"

# Create virtual environment with Python 3.12
uv venv --python 3.12 .venv
source .venv/bin/activate

# Install core Python packages
uv pip install \
    esa-snappy \
    numpy>=1.26 scipy \
    matplotlib seaborn \
    rasterio rioxarray \
    geopandas shapely pyproj \
    scikit-learn \
    jupyter ipykernel ipywidgets \
    tqdm joblib

# Register as a Jupyter kernel
.venv/bin/python -m ipykernel install --user --name sar-oilpalm --display-name "SAR-OilPalm (3.12)"
echo ""
echo "=== Environment ready ==="
echo "Kernel 'SAR-OilPalm (3.12)' registered"
echo "Run: source .venv/bin/activate"

> **Important:** After running the cell above, go to **Kernel → Change Kernel → SAR-OilPalm (3.12)** so all Python cells below use the correct environment.

### 2. Download & Install ESA SNAP 13.0.0

Downloads the Sentinel Toolboxes installer (~1 GB) and runs a quiet headless installation to `~/snap`.

In [ ]:
%%bash
set -e

INSTALLER="esa-snap_sentinel_linux-13.0.0.sh"
INSTALLER_URL="https://download.esa.int/step/snap/13.0/installers/${INSTALLER}"
SNAP_HOME="$HOME/snap"

# Download if not already present
if [ ! -f "$INSTALLER" ]; then
    echo "Downloading SNAP 13.0.0 Sentinel Toolboxes (~1 GB)..."
    wget "$INSTALLER_URL" -O "$INSTALLER"
fi
chmod +x "$INSTALLER"
echo "Installer ready: $(du -h $INSTALLER | cut -f1)"

# Create response file for quiet headless install
cat > response.varfile << 'EOF'
sys.adminRights$Boolean=false
sys.component.RSTB$Boolean=true
sys.component.S1TBX$Boolean=true
sys.component.S2TBX$Boolean=true
sys.component.S3TBX$Boolean=false
sys.component.SNAP$Boolean=true
sys.installationDir=$HOME/snap
sys.languageId=en
sys.programGroupDisabled$Boolean=true
createDesktopLinkAction$Boolean=false
executeLauncherWithPythonAction$Boolean=false
deleteSnapDir$Boolean=false
EOF

# Substitute actual home path (the heredoc literal prevents expansion)
sed -i "s|\\$HOME|$HOME|" response.varfile

# Run installer in quiet mode
mkdir -p "$SNAP_HOME"
echo "Installing SNAP to $SNAP_HOME (this may take a few minutes)..."
bash "$INSTALLER" -q -varfile response.varfile
echo "SNAP installation complete"

# Add bin to PATH permanently
if ! grep -q "$SNAP_HOME/bin" ~/.bashrc 2>/dev/null; then
    echo "export PATH=\\$PATH:$SNAP_HOME/bin" >> ~/.bashrc
fi
export PATH="$PATH:$SNAP_HOME/bin"

# Verify
ls -la "$SNAP_HOME/bin/" | head -10
echo ""
echo "=== SNAP installed at $SNAP_HOME ==="
echo "Run: export PATH=\\$PATH:$SNAP_HOME/bin"

### 3. Configure esa_snappy (SNAP ↔ Python Bridge)

Connects SNAP's Java engine to Python via `esa_snappy`, then verifies everything works.

In [ ]:
%%bash
set -e

SNAP_HOME="$HOME/snap"
VENV_PYTHON="$(pwd)/.venv/bin/python"

export PATH="$SNAP_HOME/bin:$PATH"

echo "=== Installing esa_snappy ==="
"$VENV_PYTHON" -m pip install esa-snappy

echo ""
echo "=== Running snappy-conf ==="
"$SNAP_HOME/bin/snappy-conf" "$VENV_PYTHON"

echo ""
echo "=== Verification ==="
"$VENV_PYTHON" -c "
import esa_snappy
from esa_snappy import ProductIO, GPF
print(f'esa_snappy version: {esa_snappy.__version__}')
print(f'SNAP home: {esa_snappy.get_snap_home()}')
print('OK - All imports successful')
"

echo ""
echo "=== GPT CLI ==="
"$SNAP_HOME/bin/gpt" -h 2>&1 | head -8 || true

# Bump JVM memory to 8G (70% of typical 12GB RAM)
SNAPPY_INI=$("$VENV_PYTHON" -c "import esa_snappy; import pathlib; print(pathlib.Path(esa_snappy.__file__).parent / 'esa_snappy.ini')")
if [ -f "$SNAPPY_INI" ]; then
    echo ""
    echo "=== Memory settings ==="
    grep -q 'java_max_mem' "$SNAPPY_INI" && \
        sed -i 's/^# *java_max_mem:.*/java_max_mem: 8G/' "$SNAPPY_INI" || \
        echo "java_max_mem: 8G" >> "$SNAPPY_INI"
    echo "JVM max memory set to 8G in $SNAPPY_INI"
fi

echo ""
echo "=== Setup Complete ==="

---
**Next steps:**
1. Place your ALOS-2 PALSAR-2 scenes in a `data/` directory
2. Proceed to the preprocessing section to calibrate, speckle-filter, and terrain-correct the imagery